<a href="https://colab.research.google.com/github/1pawn0/Google-Colab-Public-Notebooks/blob/main/Kaggle-Competitions/contradictory_my_dear_watson_XLM_RoBERTa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -qU torch transformers tokenizers kagglehub polars scikit-learn

In [ ]:
import os
import shutil
import sys
from pathlib import Path

import numpy as np
import polars as pl
import torch
from sklearn.metrics import accuracy_score, f1_score
from google.colab import userdata
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import (
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    XLMRobertaConfig,
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
import kagglehub

print(
    "python " + sys.version.split()[0],
    "torch " + torch.__version__,
    "polars " + pl.__version__,
    "kagglehub " + kagglehub.__version__,
    sep="\n",
)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


### Load the pretrained [`XLM-RoBERTa`](https://huggingface.co/docs/transformers/main/en/model_doc/xlm-roberta) Model

In [ ]:
from transformers import (
    Trainer,
    TrainingArguments,
    XLMRobertaConfig,
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
)

MODEL_NAME = "FacebookAI/xlm-roberta-large"
model_config = XLMRobertaConfig.from_pretrained(MODEL_NAME, num_labels=3)
tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL_NAME)
model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME, config=model_config).to(device)  # used for natural language inference (NLI)


### Downloading the competition's dataset

In [ ]:
competition_name = "contradictory-my-dear-watson"
competition_path: Path = Path(kagglehub.competition.competition_download(competition_name))
competition_files: list = os.listdir(competition_path)
print(competition_files)
competition_data_path: Path = Path(f"./data/{competition_name}")
shutil.copytree(competition_path, competition_data_path, dirs_exist_ok=True)
print(competition_data_path)

Load all csv files of the competition into polars dataframes

In [ ]:
train_df = pl.read_csv(competition_data_path / "train.csv")
test_df = pl.read_csv(competition_data_path / "test.csv")
sample_submission_df = pl.read_csv(competition_data_path / "sample_submission.csv")


### Perform EDA on the loaded dataframes

In [ ]:
print(train_df)
print(train_df.glimpse())
print(train_df.schema)
print(train_df.describe())

In [ ]:
print(test_df)
print(test_df.glimpse())
print(test_df.schema)
print(test_df.describe())

### Training

In [ ]:
from datasets import Dataset
import warnings
from transformers import logging

# Suppress the specific warning
logging.set_verbosity_error()
warnings.filterwarnings('ignore')

def tokenize_function(examples):
    return tokenizer(examples["premise"], examples["hypothesis"], truncation=True, padding="max_length", max_length=128)


train_dataset = Dataset.from_dict(
    {"premise": train_df["premise"].to_list(), "hypothesis": train_df["hypothesis"].to_list(), "label": train_df["label"].to_list()}
).map(tokenize_function, batched=True)


test_dataset = Dataset.from_dict({"premise": test_df["premise"].to_list(), "hypothesis": test_df["hypothesis"].to_list(), "id": test_df["id"].to_list()}).map(
    tokenize_function, batched=True
)


In [ ]:
from transformers import EarlyStoppingCallback
import gc

del model, trainer
torch.cuda.empty_cache()
gc.collect()

model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME, config=model_config).to("cuda")


def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1": f1}


training_args = TrainingArguments(
    output_dir="./xlm-roberta-nli",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=50,
    eval_steps=100,
    save_steps=100,
    save_strategy="steps",
    eval_strategy="steps",
    load_best_model_at_end=True,
    report_to="none",
    dataloader_pin_memory=False,
    fp16=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    max_steps=1000,
)
train_test_split = train_dataset.train_test_split(test_size=0.1, seed=42)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_test_split["train"],
    eval_dataset=train_test_split["test"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)
trainer.train(resume_from_checkpoint=False)


In [ ]:
trainer.train()

predictions = trainer.predict(test_dataset)
predicted_labels = predictions.predictions.argmax(-1)

submission_df = pl.DataFrame({"id": test_df["id"], "label": predicted_labels.tolist()})

submission_df.write_csv("submission.csv")
print(f"Submission saved with {len(submission_df)} predictions")


In [ ]:
import torch
import gc

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

# To fully free memory, delete model and trainer
del model
del trainer
torch.cuda.empty_cache()
gc.collect()

# Check memory usage
print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")